# SnowHealth Demand Forecasting - Drift Simulation & Load Test

This notebook sends **1 million inference calls** to the deployed model over ~5 minutes, progressively injecting **data drift** to trigger drift detection.

**Drift strategy:**
- **Minutes 0-1**: Clean data (baseline distribution)
- **Minutes 1-2**: Mild drift (temperature +10F, flu index +3)
- **Minutes 2-3**: Moderate drift (prices +40%, demand patterns shifted)
- **Minutes 3-4**: Severe drift (entirely out-of-distribution features)
- **Minutes 4-5**: Catastrophic drift (extreme outliers, inverted seasonality)

Each batch writes predictions + drift metadata to the monitoring source table so the Model Monitor can detect the degradation.

In [ ]:
# Initialize
import sys, os, time
work_dir = os.getcwd()
sys.path.insert(0, os.path.join(work_dir, 'source'))

from snowflake.snowpark.context import get_active_session
import config as cfg
from feature_engineering import get_feature_columns
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

session = get_active_session()
session.sql(f"USE DATABASE {cfg.SNOWHEALTH_DATABASE}").collect()
session.sql(f"USE SCHEMA {cfg.SNOWHEALTH_MODELS_SCHEMA}").collect()

# Load model
from snowflake.ml.registry import Registry
registry = Registry(session=session, database_name=cfg.SNOWHEALTH_DATABASE, schema_name=cfg.SNOWHEALTH_MODELS_SCHEMA)
model_ref = registry.get_model(cfg.SNOWHEALTH_MODEL_NAME)
mv = model_ref.version("tuned_v1")
print(f"Model loaded: {cfg.SNOWHEALTH_MODEL_NAME} v tuned_v1")
print(f"Functions: {[f['name'] for f in mv.show_functions()]}")

In [ ]:
# Load baseline data and identify feature columns
df_base = session.table(f"{cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_FEATURES_SCHEMA}.SNOWHEALTH_DEMAND_FEATURES").to_pandas()
feature_cols = get_feature_columns(df_base, cfg.TARGET_COLUMN)
X_base = df_base[feature_cols].astype(float)
y_base = df_base[cfg.TARGET_COLUMN].astype(float)

print(f"Baseline data: {len(X_base)} rows, {len(feature_cols)} features")
print(f"Feature columns: {feature_cols}")
print(f"\nBaseline feature stats:")
print(X_base.describe().round(2).to_string())

In [ ]:
def generate_drifted_batch(X_base, y_base, batch_size, drift_level, rng):
    """
    Generate a batch of data with controlled drift.
    drift_level: 0=clean, 1=mild, 2=moderate, 3=severe, 4=catastrophic
    """
    # Sample with replacement from base data
    idx = rng.choice(len(X_base), size=batch_size, replace=True)
    X_batch = X_base.iloc[idx].copy().reset_index(drop=True)
    y_batch = y_base.iloc[idx].copy().reset_index(drop=True)
    
    if drift_level == 0:
        # Clean - add small Gaussian noise only
        noise = rng.normal(0, 0.01, X_batch.shape)
        X_batch = X_batch + noise * X_batch.std().values
        return X_batch, y_batch, "clean"
    
    elif drift_level == 1:
        # Mild drift - shift temperature and flu index
        if 'TEMPERATURE_F' in X_batch.columns:
            X_batch['TEMPERATURE_F'] += rng.normal(10, 3, batch_size)
        if 'FLU_INDEX' in X_batch.columns:
            X_batch['FLU_INDEX'] += rng.normal(3, 1, batch_size)
            X_batch['FLU_INDEX'] = X_batch['FLU_INDEX'].clip(lower=0)
        if 'MONTH_SIN' in X_batch.columns:
            X_batch['MONTH_SIN'] += rng.normal(0.3, 0.1, batch_size)
        return X_batch, y_batch, "mild"
    
    elif drift_level == 2:
        # Moderate drift - prices spike, promotions disappear, inventory drops
        if 'UNIT_PRICE' in X_batch.columns:
            X_batch['UNIT_PRICE'] *= rng.uniform(1.3, 1.8, batch_size)
        if 'PROMOTION_DISCOUNT_PCT' in X_batch.columns:
            X_batch['PROMOTION_DISCOUNT_PCT'] = 0  # no promotions
        if 'PRICE_TIMES_PROMO' in X_batch.columns:
            X_batch['PRICE_TIMES_PROMO'] = 0
        if 'INVENTORY_ON_HAND' in X_batch.columns:
            X_batch['INVENTORY_ON_HAND'] = (X_batch['INVENTORY_ON_HAND'] * 0.3).astype(int)
        if 'DAYS_SUPPLY_ON_HAND' in X_batch.columns:
            X_batch['DAYS_SUPPLY_ON_HAND'] *= 0.3
        if 'TEMPERATURE_F' in X_batch.columns:
            X_batch['TEMPERATURE_F'] += rng.normal(20, 5, batch_size)
        if 'FLU_INDEX' in X_batch.columns:
            X_batch['FLU_INDEX'] += rng.normal(5, 2, batch_size)
        return X_batch, y_batch, "moderate"
    
    elif drift_level == 3:
        # Severe drift - out-of-distribution features
        if 'UNIT_PRICE' in X_batch.columns:
            X_batch['UNIT_PRICE'] *= rng.uniform(2.0, 5.0, batch_size)
        if 'TEMPERATURE_F' in X_batch.columns:
            X_batch['TEMPERATURE_F'] = rng.uniform(100, 130, batch_size)  # extreme heat
        if 'FLU_INDEX' in X_batch.columns:
            X_batch['FLU_INDEX'] = rng.uniform(12, 20, batch_size)  # pandemic-level
        if 'INVENTORY_ON_HAND' in X_batch.columns:
            X_batch['INVENTORY_ON_HAND'] = rng.choice([0, 1, 2], batch_size)  # stockout
        if 'DAYS_SUPPLY_ON_HAND' in X_batch.columns:
            X_batch['DAYS_SUPPLY_ON_HAND'] = rng.uniform(0, 0.5, batch_size)
        if 'SEASONAL_DEMAND_INDEX' in X_batch.columns:
            X_batch['SEASONAL_DEMAND_INDEX'] = rng.uniform(2.0, 4.0, batch_size)
        if 'COMPETITOR_PROXIMITY_MILES' in X_batch.columns:
            X_batch['COMPETITOR_PROXIMITY_MILES'] = rng.uniform(0.01, 0.1, batch_size)
        # Invert cyclical features
        for col in ['MONTH_SIN', 'MONTH_COS', 'DOW_SIN', 'DOW_COS']:
            if col in X_batch.columns:
                X_batch[col] = -X_batch[col]
        return X_batch, y_batch, "severe"
    
    else:  # drift_level >= 4
        # Catastrophic drift - random noise, completely synthetic
        for col in X_batch.columns:
            col_std = X_base[col].std()
            col_mean = X_base[col].mean()
            # Shift mean by 5 standard deviations and add heavy noise
            X_batch[col] = rng.normal(col_mean + 5 * col_std, col_std * 3, batch_size)
        if 'UNIT_PRICE' in X_batch.columns:
            X_batch['UNIT_PRICE'] = X_batch['UNIT_PRICE'].clip(lower=0.01)
        if 'INVENTORY_ON_HAND' in X_batch.columns:
            X_batch['INVENTORY_ON_HAND'] = X_batch['INVENTORY_ON_HAND'].clip(lower=0).astype(int)
        return X_batch, y_batch, "catastrophic"

print("Drift generator ready")
print("Levels: 0=clean, 1=mild, 2=moderate, 3=severe, 4=catastrophic")

In [ ]:
# ===== DRIFT SIMULATION CONFIG =====
TOTAL_INFERENCES = 1_000_000
BATCH_SIZE = 50_000  # rows per batch inference call
TOTAL_BATCHES = TOTAL_INFERENCES // BATCH_SIZE  # 20 batches
SIMULATION_MINUTES = 5
SECONDS_PER_BATCH = (SIMULATION_MINUTES * 60) / TOTAL_BATCHES  # ~15 sec per batch

# Drift schedule: which batches get which drift level
# Batches 1-4: clean, 5-8: mild, 9-12: moderate, 13-16: severe, 17-20: catastrophic
batches_per_phase = TOTAL_BATCHES // 5
drift_schedule = (
    [0] * batches_per_phase +   # clean
    [1] * batches_per_phase +   # mild
    [2] * batches_per_phase +   # moderate
    [3] * batches_per_phase +   # severe
    [4] * (TOTAL_BATCHES - 4 * batches_per_phase)  # catastrophic (remaining)
)

print(f"=" * 60)
print(f"DRIFT SIMULATION PLAN")
print(f"=" * 60)
print(f"Total inferences:     {TOTAL_INFERENCES:,}")
print(f"Batch size:           {BATCH_SIZE:,}")
print(f"Total batches:        {TOTAL_BATCHES}")
print(f"Target duration:      {SIMULATION_MINUTES} minutes")
print(f"Seconds per batch:    ~{SECONDS_PER_BATCH:.0f}s")
print(f"")
print(f"Drift Schedule:")
phase_names = ['Clean', 'Mild Drift', 'Moderate Drift', 'Severe Drift', 'Catastrophic Drift']
for i, name in enumerate(phase_names):
    batch_start = i * batches_per_phase + 1
    batch_end = min((i + 1) * batches_per_phase, TOTAL_BATCHES)
    inferences = (batch_end - batch_start + 1) * BATCH_SIZE
    print(f"  Batches {batch_start:2d}-{batch_end:2d}: {name:20s} ({inferences:,} inferences)")
print(f"=" * 60)

In [ ]:
# ===== RUN THE SIMULATION =====
rng = np.random.default_rng(42)

results_log = []
total_inferences = 0
start_time = time.time()

print(f"Starting drift simulation at {datetime.now().strftime('%H:%M:%S')}")
print(f"{'Batch':>6} {'Drift':>14} {'Rows':>8} {'Cumul':>10} {'AvgPred':>9} {'StdPred':>9} {'Elapsed':>8} {'Status'}")
print("-" * 85)

for batch_num in range(TOTAL_BATCHES):
    batch_start = time.time()
    drift_level = drift_schedule[batch_num]
    
    # Generate drifted data
    X_batch, y_batch, drift_label = generate_drifted_batch(
        X_base, y_base, BATCH_SIZE, drift_level, rng
    )
    
    # Run batch inference through the model
    predictions = mv.run(X_batch)
    pred_values = predictions.iloc[:, 0].values
    
    total_inferences += BATCH_SIZE
    elapsed = time.time() - start_time
    
    # Compute batch statistics
    avg_pred = np.mean(pred_values)
    std_pred = np.std(pred_values)
    
    # Write predictions + drift metadata to monitoring source table
    monitor_batch = X_batch.copy()
    monitor_batch['PREDICTION'] = pred_values
    monitor_batch['UNITS_SOLD'] = y_batch.values  # actual label (from original data)
    monitor_batch['TIMESTAMP'] = pd.Timestamp.now() + pd.Timedelta(seconds=batch_num * 15)
    monitor_batch['DRIFT_LEVEL'] = drift_level
    monitor_batch['DRIFT_LABEL'] = drift_label
    monitor_batch['BATCH_NUM'] = batch_num + 1
    
    # Convert types for Snowflake
    for col in monitor_batch.select_dtypes(include=['int8', 'int16', 'int32']).columns:
        monitor_batch[col] = monitor_batch[col].astype('int64')
    
    snow_batch = session.create_dataframe(monitor_batch)
    snow_batch.write.mode('append').save_as_table(
        f"{cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}.SNOWHEALTH_DRIFT_SIMULATION_LOG"
    )
    
    # Log
    batch_elapsed = time.time() - batch_start
    results_log.append({
        'batch': batch_num + 1,
        'drift_level': drift_level,
        'drift_label': drift_label,
        'rows': BATCH_SIZE,
        'avg_pred': round(avg_pred, 2),
        'std_pred': round(std_pred, 2),
        'batch_time': round(batch_elapsed, 1),
    })
    
    status_emoji = ['OK', 'MILD', 'MODERATE', 'SEVERE', 'CATASTROPHIC'][drift_level]
    print(f"{batch_num+1:6d} {drift_label:>14} {BATCH_SIZE:>8,} {total_inferences:>10,} {avg_pred:>9.2f} {std_pred:>9.2f} {elapsed:>7.1f}s  {status_emoji}")
    
    # Pace to spread over 5 minutes
    target_time = (batch_num + 1) * SECONDS_PER_BATCH
    if elapsed < target_time:
        sleep_time = target_time - elapsed
        if sleep_time > 0.5:
            time.sleep(sleep_time)

total_elapsed = time.time() - start_time
print("-" * 85)
print(f"SIMULATION COMPLETE: {total_inferences:,} inferences in {total_elapsed:.1f}s ({total_elapsed/60:.1f} min)")

In [ ]:
# ===== ANALYZE DRIFT IMPACT =====
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

results_df = pd.DataFrame(results_log)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Average prediction by batch (drift progression)
colors = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c', '#8e44ad']
batch_colors = [colors[d] for d in results_df['drift_level']]
axes[0,0].bar(results_df['batch'], results_df['avg_pred'], color=batch_colors, edgecolor='white')
axes[0,0].set_xlabel('Batch Number')
axes[0,0].set_ylabel('Average Prediction')
axes[0,0].set_title('Average Prediction by Batch', fontsize=13, fontweight='bold')
axes[0,0].axhline(y=X_base.iloc[:, 0].mean() if len(feature_cols) > 0 else 33, color='black', linestyle='--', alpha=0.5, label='Baseline mean')

# 2. Prediction std deviation (volatility increases with drift)
axes[0,1].bar(results_df['batch'], results_df['std_pred'], color=batch_colors, edgecolor='white')
axes[0,1].set_xlabel('Batch Number')
axes[0,1].set_ylabel('Prediction Std Dev')
axes[0,1].set_title('Prediction Volatility by Batch', fontsize=13, fontweight='bold')

# 3. Average prediction by drift level
drift_avg = results_df.groupby('drift_label')[['avg_pred', 'std_pred']].mean()
drift_order = ['clean', 'mild', 'moderate', 'severe', 'catastrophic']
drift_avg = drift_avg.reindex(drift_order)
axes[1,0].bar(range(len(drift_avg)), drift_avg['avg_pred'], color=colors[:len(drift_avg)], edgecolor='white')
axes[1,0].set_xticks(range(len(drift_avg)))
axes[1,0].set_xticklabels(['Clean', 'Mild', 'Moderate', 'Severe', 'Catastrophic'])
axes[1,0].set_ylabel('Average Prediction')
axes[1,0].set_title('Avg Prediction by Drift Level', fontsize=13, fontweight='bold')

# 4. Batch processing time
axes[1,1].plot(results_df['batch'], results_df['batch_time'], 'o-', color='#cc0000', markersize=5)
axes[1,1].set_xlabel('Batch Number')
axes[1,1].set_ylabel('Processing Time (seconds)')
axes[1,1].set_title('Batch Processing Time', fontsize=13, fontweight='bold')
axes[1,1].axhline(y=results_df['batch_time'].mean(), color='gray', linestyle='--', alpha=0.5)

plt.suptitle('SnowHealth Demand Forecast - Drift Simulation Results (1M Inferences)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ===== QUERY DRIFT DETECTION FROM SIMULATION LOG =====
print("=" * 70)
print("DRIFT DETECTION ANALYSIS")
print("=" * 70)

# Calculate error metrics by drift level from the simulation log
drift_analysis = session.sql(f"""
    SELECT
        DRIFT_LABEL,
        DRIFT_LEVEL,
        COUNT(*) AS ROW_COUNT,
        ROUND(AVG(ABS(UNITS_SOLD - PREDICTION)), 4) AS MAE,
        ROUND(SQRT(AVG(POWER(UNITS_SOLD - PREDICTION, 2))), 4) AS RMSE,
        ROUND(AVG(ABS(UNITS_SOLD - PREDICTION) / GREATEST(UNITS_SOLD, 1)) * 100, 2) AS MAPE,
        ROUND(AVG(PREDICTION), 2) AS AVG_PREDICTION,
        ROUND(STDDEV(PREDICTION), 2) AS PREDICTION_STD,
        ROUND(AVG(UNITS_SOLD), 2) AS AVG_ACTUAL,
        MIN(TIMESTAMP) AS FIRST_TS,
        MAX(TIMESTAMP) AS LAST_TS
    FROM {cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}.SNOWHEALTH_DRIFT_SIMULATION_LOG
    GROUP BY DRIFT_LABEL, DRIFT_LEVEL
    ORDER BY DRIFT_LEVEL
""").to_pandas()

print("\nError Metrics by Drift Level:")
print(drift_analysis[['DRIFT_LABEL', 'ROW_COUNT', 'MAE', 'RMSE', 'MAPE', 'AVG_PREDICTION', 'AVG_ACTUAL']].to_string(index=False))

print("\n" + "-" * 70)
print("DRIFT DETECTION SIGNALS:")
print("-" * 70)
baseline_rmse = drift_analysis.loc[drift_analysis['DRIFT_LEVEL'] == 0, 'RMSE'].values[0]
for _, row in drift_analysis.iterrows():
    rmse_change = ((row['RMSE'] - baseline_rmse) / baseline_rmse) * 100
    pred_shift = abs(row['AVG_PREDICTION'] - row['AVG_ACTUAL'])
    alert = 'ALERT' if row['RMSE'] > 6.0 else 'OK'
    print(f"  {row['DRIFT_LABEL']:>14}: RMSE={row['RMSE']:>8.4f} ({rmse_change:>+7.1f}%)  "
          f"Pred shift={pred_shift:>6.2f}  [{alert}]")

In [ ]:
# ===== PREDICTION DISTRIBUTION SHIFT ANALYSIS =====
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Pull prediction distributions by drift level
for drift_level in range(5):
    preds = session.sql(f"""
        SELECT PREDICTION FROM {cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}.SNOWHEALTH_DRIFT_SIMULATION_LOG
        WHERE DRIFT_LEVEL = {drift_level}
        ORDER BY RANDOM()
        LIMIT 10000
    """).to_pandas()['PREDICTION'].values
    
    label = ['Clean', 'Mild', 'Moderate', 'Severe', 'Catastrophic'][drift_level]
    color = colors[drift_level]
    axes[0].hist(preds, bins=50, alpha=0.4, color=color, label=label, density=True)
    
    # Residuals
    resid = session.sql(f"""
        SELECT (UNITS_SOLD - PREDICTION) AS RESIDUAL 
        FROM {cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}.SNOWHEALTH_DRIFT_SIMULATION_LOG
        WHERE DRIFT_LEVEL = {drift_level}
        ORDER BY RANDOM()
        LIMIT 10000
    """).to_pandas()['RESIDUAL'].values
    axes[1].hist(resid, bins=50, alpha=0.4, color=color, label=label, density=True)

axes[0].set_title('Prediction Distribution by Drift Level', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Predicted Units')
axes[0].legend()

axes[1].set_title('Residual Distribution by Drift Level', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Residual (Actual - Predicted)')
axes[1].axvline(x=0, color='black', linestyle='--', linewidth=1)
axes[1].legend()

plt.suptitle('Distribution Shift Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ===== FINAL SUMMARY =====
total_rows = session.sql(f"""
    SELECT COUNT(*) AS CNT FROM {cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}.SNOWHEALTH_DRIFT_SIMULATION_LOG
""").to_pandas()['CNT'].values[0]

print("=" * 70)
print("DRIFT SIMULATION COMPLETE")
print("=" * 70)
print(f"")
print(f"Total inferences sent:       {total_rows:,}")
print(f"Simulation duration:          {total_elapsed:.1f}s ({total_elapsed/60:.1f} min)")
print(f"Throughput:                   {total_rows/total_elapsed:,.0f} inferences/sec")
print(f"")
print(f"DRIFT DETECTION RESULTS:")
for _, row in drift_analysis.iterrows():
    alert = 'DRIFT DETECTED' if row['RMSE'] > 6.0 else 'Within threshold'
    print(f"  {row['DRIFT_LABEL']:>14}: RMSE={row['RMSE']:.4f}, MAPE={row['MAPE']:.1f}% -> {alert}")
print(f"")
print(f"Alert threshold: RMSE > 6.0")
baseline_rmse = drift_analysis.loc[drift_analysis['DRIFT_LEVEL'] == 0, 'RMSE'].values[0]
print(f"Baseline RMSE:   {baseline_rmse:.4f}")
max_rmse = drift_analysis['RMSE'].max()
print(f"Max RMSE:        {max_rmse:.4f} ({max_rmse/baseline_rmse:.1f}x baseline)")
print(f"")
print(f"Simulation log:  {cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}.SNOWHEALTH_DRIFT_SIMULATION_LOG")
print(f"Monitor source:  {cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}.SNOWHEALTH_DEMAND_MONITOR_SOURCE")
print("=" * 70)